In [2]:
"""
MODELO 2/12 — Support Vector Machine (SVM)

Replica _IM_SVM.R:
    - Lê {Code}_DatasetNew.csv (features já normalizadas)
    - WFA: d1=250 (treino), d2=5 (predição), janela deslizante
    - kernlab::ksvm(kernel="rbfdot", scaled=F, type="C-svc", C=10)
    - predict retorna classes (1 ou 2), converte para (0 ou 1) via -1
    - Salva {Code}_TradeSignal_SVM.csv

Nota: o R kernlab retorna classes 1 e 2 (factor levels), por isso o
código original faz predict_signal-1. No sklearn, SVC já retorna 0 e 1
quando treinado com labels 0/1.

Uso:
    python 04_model_SVM.py
"""

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# ===================== CONFIGURAÇÃO =====================
BASE_DIR = Path(r"C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICSMI")
SEC_NAMES = BASE_DIR / ".NewB3_pruned.csv"

TRAIN_SIZE = 250
TEST_SIZE = 5
C_VALUE = 10       # C=10 no código R (apesar do comentário dizer C=1)
# ========================================================


def read_codes(path: Path) -> list:
    df = pd.read_csv(path, dtype=str, encoding="utf-8-sig")
    return df["Code"].str.strip().str.upper().tolist()


def run_wfa_svm(code: str, base_dir: Path) -> dict:
    """Executa Walk-Forward Analysis com SVM para um ticker."""
    infile = base_dir / f"{code}_DatasetNew_MI.csv"
    outfile = base_dir / f"{code}_TradeSignal_SVM_MI.csv"

    if outfile.exists():
        return {"Code": code, "status": "skipped", "signals": 0}

    if not infile.exists():
        return {"Code": code, "status": "no_DatasetNew", "signals": 0}

    try:
        df = pd.read_csv(infile, encoding="utf-8-sig")
    except Exception as e:
        return {"Code": code, "status": f"read_error: {e}", "signals": 0}

    if df.shape[1] < 3:
        return {"Code": code, "status": "too_few_columns", "signals": 0}

    date_col = df.columns[0]
    label_col = df.columns[-1]
    feature_cols = df.columns[1:-1].tolist()

    # --- Alinhamento WFA (idêntico ao R) ---
    M = len(df)
    if M < TRAIN_SIZE + TEST_SIZE:
        return {"Code": code, "status": f"too_few_rows ({M})", "signals": 0}

    Q = (M - TRAIN_SIZE) // TEST_SIZE
    H = (M - TRAIN_SIZE) - TEST_SIZE * Q
    df = df.iloc[H:].reset_index(drop=True)

    dates = df[date_col].values
    X = df[feature_cols].values.astype(float)
    y = df[label_col].values.astype(int)

    predict_signal = []
    predict_dates = []

    # --- Loop WFA ---
    for i in range(Q):
        train_start = i * TEST_SIZE
        train_end = train_start + TRAIN_SIZE
        test_start = train_end
        test_end = test_start + TEST_SIZE

        X_train = X[train_start:train_end]
        y_train = y[train_start:train_end]
        X_test = X[test_start:test_end]
        test_dates = dates[test_start:test_end]

        if len(np.unique(y_train)) < 2:
            preds = [int(y_train[0])] * len(X_test)
        else:
            try:
                # kernlab::ksvm(kernel="rbfdot", scaled=F, type="C-svc", C=10)
                # rbfdot = kernel RBF
                # scaled=F = dados já normalizados, não escalar novamente
                clf = SVC(
                    kernel="rbf",
                    C=C_VALUE,
                    probability=False,
                )
                clf.fit(X_train, y_train)
                preds = clf.predict(X_test).astype(int).tolist()
            except Exception:
                preds = [0] * len(X_test)

        predict_signal.extend(preds)
        predict_dates.extend(test_dates)

    # --- Salvar ---
    if predict_signal:
        df_out = pd.DataFrame({"Date": predict_dates, "pre_signal": predict_signal})
        df_out.to_csv(outfile, index=False, encoding="utf-8-sig")

    return {"Code": code, "status": "ok", "signals": len(predict_signal)}


def main():
    codes = read_codes(SEC_NAMES)
    print(f"Modelo: SVM (kernel=RBF, C={C_VALUE})")
    print(f"WFA: d1={TRAIN_SIZE}, d2={TEST_SIZE}")
    print(f"Tickers: {len(codes)}\n")

    report = []
    for code in tqdm(codes, desc="SVM Walk-Forward"):
        result = run_wfa_svm(code, BASE_DIR)
        report.append(result)

    report_df = pd.DataFrame(report)
    report_df.to_csv(BASE_DIR / "model_SVM_report.csv", index=False, encoding="utf-8-sig")

    n_ok = (report_df["status"] == "ok").sum()
    n_skip = (report_df["status"] == "skipped").sum()
    avg_signals = report_df.loc[report_df["status"] == "ok", "signals"].mean()

    print(f"\n{'='*50}")
    print(f"Concluído: {n_ok} processados, {n_skip} já existiam.")
    print(f"Média de sinais por ação: {avg_signals:.0f}")
    print(f"Relatório: model_SVM_report.csv")


if __name__ == "__main__":
    main()

Modelo: SVM (kernel=RBF, C=10)
WFA: d1=250, d2=5
Tickers: 239



SVM Walk-Forward: 100%|██████████| 239/239 [02:08<00:00,  1.86it/s]


Concluído: 239 processados, 0 já existiam.
Média de sinais por ação: 564
Relatório: model_SVM_report.csv
